# Deliverable 1.1 Domain Models

## Learner Domain Model

In [15]:
import re
from datetime import datetime, timezone, timedelta

south_africa_tz = timezone(timedelta(hours=2))

class Learner:
    # represents a learner enolled

    # maintains the list of course reg and support tickets associayed with the learner.

    _id_counter = 1
    EMAIL_REGEX = re.compile(r'^[\w\-\.]+@([\w\-]+\.)+[\w\-]{2,4}$') # https://regexr.com/3e48o

    def __init__(self, name, email, phone=None):
        if not name or not isinstance(name, str):
            raise ValueError("Learner name is needed and must be a string")
        if not email or not self.EMAIL_REGEX.match(email):
            raise ValueError(f"Invalid email address: {repr(email)}")

        self.learner_id = f"L{Learner._id_counter:03d}" # pad with zeros ad make 3 characters wide
        Learner._id_counter += 1

        self.name = name
        self.email = email
        self.phone = phone
        self.date_joined = datetime.now(south_africa_tz)
        self.registrations = []
        self.support_tickets = []

    def __repr__(self):
        return f"<Learner {self.learner_id}x{self.name}>" # matches teh style of memory addresses 

    def __str__(self):
        return f"Learner ID: {self.learner_id}, Name: {self.name}" # for when we print the class

    def register_for_course(self, course):
        # register learner via a course enrolment function in course class
        if self.has_registration_for(course):
            raise ValueError(f"Learner {self.learner_id} is already registered for {course.course_code}")

        registration = course.enroll(self)
        self.registrations.append(registration)
        return registration

    def has_registration_for(self, course):
        return any(r.course.course_code == course.course_code for r in self.registrations)

    def raise_support_ticket(self, ticket):
        self.support_tickets.append(ticket)

    def get_avrg_score(self):
        scores = [reg.assessment.get_percentage() for reg in self.registrations if reg.assessment is not None]

        if not scores: return None
        return round(sum(scores)/len(scores), 2)

## Course domain model

In [16]:
class Course:
    # enforces enrolment capacity and tracks registrations for a course
    def __init__(self, course_code, title, capacity=50, instructor=None):
        if not course_code:
            raise ValueError("Needs Course code")
        if not title:
            raise ValueError("Needs a title")
        if capacity <= 0:
            raise ValueError("Capacity must be greater than zero")

        self.course_code = course_code
        self.title = title
        self.capacity = capacity
        self.instructor = instructor
        self.registrations = []

    @property
    def enrolled_count(self) -> int:
        return len([r for r in self.registrations if r.status != "CANCELLED"])

    @property
    def is_full(self) -> bool:
        return self.enrolled_count >= self.capacity

    def enroll(self, learner):
        # create a registration and enforce capacity rules
        if self.is_full:
            raise ValueError(f"Course {self.course_code} is full. annot register {learner.name}")

        registration = Registration(learner, self)
        self.registrations.append(registration)
        return registration

    def cancel_registration(self, registration):
        # cancel a registration
        if registration not in self.registrations:
            raise ValueError("This reg does not beling to this course")
        registration.status = "CANCELLED"

    def __repr__(self):
        return f"<Course {self.course_code}x{self.title}>"

    def __str__(self):
        return f"Course: {self.course_code} - {self.title}"

## Registration domain model

In [17]:
from datetime import datetime, timezone, timedelta

south_africa_tz = timezone(timedelta(hours=2))

class Registration:
    # reps one learners enrolment into one course using its own state and linked assessment
    _id_counter = 1
    VALID_STATUSES = {"PENDING", "CONFIRMED", "CANCELLED", "COMPLETED"}

    def __init__(self, learner, course, status="CONFIRMED"):
        if learner is None or course is None:
            raise ValueError("A registration needs a learner and a course")
        if status not in self.VALID_STATUSES:
            raise ValueError(f"Invalid status: {repr(status)}")

        self.registration_id = f"R{Registration._id_counter:04d}"
        Registration._id_counter += 1

        self.learner = learner
        self.course = course
        self.status = status
        self.date_registred = datetime.now(south_africa_tz)
        self.assessment = None

    def attach_assessment(self, assessment):
        self.assessment = assessment
        self.status = "COMPLETED"

    def cancel(self):
        self.status = "CANCELLED"

    def __repr__(self):
        return f"<Registration {self.registration_id}: {self.learner.name} -> {self.course.course_code} [{self.status}]>"

    def __str__(self):
        return f"{self.learner.name} registered for {self.course.title}"

## Assessment Domain Model

In [18]:
class Assessment:
    def __init__(self, registration, raw_score, max_score=100, strategy=None):
        if registration is None:
            raise ValueError("Assessment needs to link to a registration")
        if max_score <= 0:
            raise ValueError("Max score must be greater than zero")

        if not (0 <= raw_score <= max_score):
            raise ValueError(f"raw score must be between 0 and {max_score}")

        self.registration = registration
        self.raw_score = raw_score
        self.max_score = max_score
        self._calculator = AssessmentCalculator(strategy or PercentageStrategy())
        registration.attach_assessment(self)

    def set_strategy(self, strategy):
        self._calculator.set_strategy(strategy)

    def get_percentage(self):
        return self._calculator.calculate(self.raw_score, self.max_score)["percentage"]

    def get_classification(self):
        return self._calculator.calculate(self.raw_score, self.max_score)["classification"]

    def __repr__(self):
        return f"<Assessment {self.registration.registration_id}x{self.get_percentage()}%>"

    def __str__(self):
        return f"Score: {self.get_percentage()}% ({self.get_classification()})"


## Support Ticket Domain Model

In [19]:
from datetime import datetime, timezone, timedelta

south_africa_tz = timezone(timedelta(hours=2))

class SupportTicket:
    _id_count = 1
    VALID_PRIORITIES = {"LOW","MEDIUM","HIGH"}
    VALID_STATUSES = {"OPEN", "IN_PROGRESS", "RESOLVED", "CLOSED"}

    def __init__(self, learner, subject: str, description: str, priority="MEDIUM"):
        if learner is None:
            raise ValueError("A support ticket needs a learner")
        if not subject:
            raise ValueError("Subject is required")
        if priority not in self.VALID_PRIORITIES:
            raise ValueError(f"Invalid priority: {priority}")

        self.ticket_id = f"T{SupportTicket._id_count:04d}"
        SupportTicket._id_count += 1

        self.learner = learner
        self.subject = subject
        self.description = description
        self.priority = priority
        self.status = "OPEN"
        self.created_at = datetime.now(south_africa_tz)

        learner.raise_support_ticket(self)

    def resolve(self):
        self.status = "RESOLVED"

    def close(self):
        self.status = "CLOSED"

    def __repr__(self):
        return f"<SupportTicker {self.ticket_id}: {self.subject} [{self.status}]>"

    def __str__(self):
        return f"Ticket {self.ticket_id} - {self.subject} ({self.status}) \nDiscription:\n\t{self.description}"

# Deliverable 1.2

## Singleton Patter config manager

In [20]:
class ConfigManager:
    _instance = None

    def __new__(cls, *args, **keyword_args):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance._initialised = False
        return cls._instance

    def __init__(self):
        if self._initialised:
            return

        self._settings = {
            "app_name": "Enterprise Python Formative",
            "default_course_capacity": 30,
            "pass_mark": 50,
            "distinction_mark": 75,
        }

        self._initialised = True

    def get(self, key, default=None):
        return self._settings.get(key, default)

    def set(self, key, value):
        self._settings[key] = value

    def all_settings(self):
        return dict(self._settings)


    @classmethod
    def reset(cls):
        # clears singleton instance
        cls._instance = None

    def __repr__(self):
        return f"<ConfigManager id={id(self)}>"

## Factory Pattern for tickets

In [21]:
class AcademicTicket(SupportTicket):

    CATEGORY = "Academic"

    def __init__(self, learner, subject, description, priority="MEDIUM"):
        super().__init__(learner, subject, description, priority)
        self.category = self.CATEGORY

class TechnicalTicket(SupportTicket):
    CATEGORY = "Technical"
    
    def __init__(self, learner, subject, description, priority="MEDIUM"):
        super().__init__(learner, subject, description, priority)
        self.category = self.CATEGORY

class RegistrationTicket(SupportTicket):
    CATEGORY = "Registration"
    
    def __init__(self, learner, subject, description, priority="MEDIUM"):
        super().__init__(learner, subject, description, priority)
        self.category = self.CATEGORY


class SupportTicketFactory:

    _ticket_types = {
        "academic": AcademicTicket,
        "technical": TechnicalTicket,
        "registration": RegistrationTicket
    }

    @classmethod
    def create_ticket(cls, ticket_type, learner, subject, description, priority="MEDIUM"):
        key = ticket_type.strip().lower()
        ticket_class = cls._ticket_types.get(key)

        if ticket_class is None:
            valid = ", ".join(cls._ticket_types.keys())
            raise ValueError(f"Unknown ticket type: {repr(ticket_type)} use these types {valid}")

        return ticket_class(learner, subject, description, priority)

    @classmethod
    def register_ticket_type(cls, type_key, ticket_class):
        # Allows new ticket types to be plugged in at runtime
        if not issubclass(ticket_class, SupportTicket):
            raise TypeError("Ticket class must be a subclass of SupportTicket")

        cls._ticket_types[type_key.strip().lower()] = ticket_class

    @classmethod
    def available_types(cls):
        return list(cls._ticket_types.keys())

## Strategy patern

In [22]:
from abc import ABC, abstractmethod

class AssessmentStrategy(ABC):
    @abstractmethod
    def calculate(self, raw_score, max_score):
        # return percetage:float, classification: str
        raise NotImplementedError

class PercentageStrategy(AssessmentStrategy):
    # srandard approach used for full cred courses
    # mark >= 75 Distinction mark >= 50 pass, else fail

    DISTINCTION_THRESHOLD = 75
    PASS_THRESHOLD = 50

    def calculate(self, raw_score, max_score):
        percentage = round((raw_score/max_score)* 100, 2)

        if percentage >= self.DISTINCTION_THRESHOLD:
            classification = "Distinction"
        elif percentage >= self.PASS_THRESHOLD:
            classification = "pass"
        else:
            classification = "Fail"

        return {"percentage": percentage, "classification": classification}

class PassFailStrategy(AssessmentStrategy):
    PASS_THRESHOLD = 50

    def calculate(self, raw_score, max_score):
        percentage = round((raw_score/max_score)* 100, 2)
        classification = "Pass" if percentage >= self.PASS_THRESHOLD else "Fail"
        return {"percentage": percentage, "classification":classification}


class WeightedStrategy(AssessmentStrategy):
    DISTINCTION_THRESHOLD = 75
    PASS_THRESHOLD = 50

    def __init__(self, weight=1.0):
        if not (0 < weight <= 1):
            raise ValueError("weight must be greater than 0 but less than or equal to 1")
        self.weight = weight

    def calculate(self, raw_score, max_score):
        percentage = round((raw_score/max_score)* 100, 2)
        weighted_percentage = round(percentage * self.weight, 2)

        if percentage >= self.DISTINCTION_THRESHOLD:
            classification = "Distinction"
        elif percentage >= self.PASS_THRESHOLD:
            classification = "pass"
        else:
            classification = "Fail"

        return {"percentage": weighted_percentage, "classification": classification}


class AssessmentCalculator:
    def __init__(self, strategy: AssessmentStrategy = None):
        self._strategy = strategy or PercentageStrategy()

    def set_strategy(self, strategy: AssessmentStrategy):
        self._strategy = strategy

    def calculate(self, raw_score, max_score):
        return self._strategy.calculate(raw_score, max_score)

# Deliverable 1 outputs

In [23]:
def deliverable1_1():
    print(f'{"=" * 70}')
    print("ENTERPRISE DOMAIN MODEL DEMONSTRATION")
    print(f'{"=" * 70}')

    print("\n\nDOMAIN MODEL OUTPUT")
    print(f'{"=" * 70}\n')

    learner = Learner("Ndaedzo Mudau", "ndaedzo_mudau@ndaedzo.com")
    course = Course("ITEPA3-33", "Enterprise Python Development", capacity=16)
    print(learner)
    print(course)

    reg = learner.register_for_course(course)
    print(reg)

    print("\n")
    assessment = Assessment(reg, raw_score=82)
    print(assessment)
    print(repr(assessment))
    print(reg.status)

    print("\n")
    ticket = SupportTicket(learner, "Login issue", "Cannot access myLMS")
    print(ticket)
    print(learner.support_tickets)


def deliverable1_2():
    print(f'\n\n{"=" * 70}')
    print("DESIGN PATTERN DEMONSTRATION")
    print(f'{"=" * 70}\n')

    print("SINGLETON PATTERN DEMONSTRATION")
    print(f'{"-" * 60}')
    config1 = ConfigManager()
    config2 = ConfigManager()
    print(f"Config 1 ID: {id(config1)}")
    print(f"Config 2 ID: {id(config2)}")
    print(f"Singleton Successful: {'Same Instance' if config1 is config2 else 'Different Instances'}")
    config1.set("pass_mark", 60)
    print(f"Pass mark seen via config2 after updating config1: {config2.get('pass_mark')}")

    print(f"\n\nFACTORY PATTERN DEMONSTRATION")
    print(f'{"-" * 60}')
    learner = Learner("Naledi Khumalo", "naledi@example.com")

    academic_ticket = SupportTicketFactory.create_ticket(
        "academic", learner, "Assignment clarity", "Need clarity on Assignment 2 rubric"
    )
    print(f"Created: {academic_ticket.__class__.__name__}")

    technical_ticket = SupportTicketFactory.create_ticket(
        "technical", learner, "Cannot login", "Password reset link is not arriving", priority="HIGH"
    )
    print(f"Created: {technical_ticket.__class__.__name__}")

    registration_ticket = SupportTicketFactory.create_ticket(
        "registration", learner, "Enrolment query", "Unsure if my registration was confirmed"
    )
    print(f"Created: {registration_ticket.__class__.__name__}")

    print(f"\n\nSTRATEGY PATTERN DEMONSTRATION")
    print(f'{"-" * 60}')
    course = Course("PY701", "Enterprise Python Development", capacity=16)
    reg = learner.register_for_course(course)

    assessment = Assessment(reg, raw_score=82, strategy=PercentageStrategy())
    print(f"Percentage Result: {assessment.get_percentage()}")
    print(f"Classification Result: {assessment.get_classification()}")

    print(f"\nSwapping to PassFailStrategy at runtime:")
    assessment.set_strategy(PassFailStrategy())
    print(f"Classification Result (Pass/Fail strategy): {assessment.get_classification()}")

    print(f"\nSwapping to WeightedStrategy (30% weighting) at runtime:")
    assessment.set_strategy(WeightedStrategy(weight=0.3))
    print(f"Weighted Percentage Result: {assessment.get_percentage()}")

In [24]:
deliverable1_1()
deliverable1_2()

ENTERPRISE DOMAIN MODEL DEMONSTRATION


DOMAIN MODEL OUTPUT

Learner ID: L001, Name: Ndaedzo Mudau
Course: ITEPA3-33 - Enterprise Python Development
Ndaedzo Mudau registered for Enterprise Python Development


Score: 82.0% (Distinction)
<Assessment R0001x82.0%>
COMPLETED


Ticket T0001 - Login issue (OPEN) 
Discription:
	Cannot access myLMS
[<SupportTicker T0001: Login issue [OPEN]>]


DESIGN PATTERN DEMONSTRATION

SINGLETON PATTERN DEMONSTRATION
------------------------------------------------------------
Config 1 ID: 2387474122128
Config 2 ID: 2387474122128
Singleton Successful: Same Instance
Pass mark seen via config2 after updating config1: 60


FACTORY PATTERN DEMONSTRATION
------------------------------------------------------------
Created: AcademicTicket
Created: TechnicalTicket
Created: RegistrationTicket


STRATEGY PATTERN DEMONSTRATION
------------------------------------------------------------
Percentage Result: 82.0
Classification Result: Distinction

Swapping to PassFail

# Deliverable 2

In [25]:
"""
Registration process engine

Coordinates bulk processing of learber course-registration requests
Enforced business rules through the domain models while making sure a single failed request does not stop processing the rest of the stuff
"""
import time
import threading
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from typing import List, Optional
from datetime import datetime, timezone, timedelta

south_africa_tz = timezone(timedelta(hours=2))


@dataclass
class RegistrationRequest:
    # one learner into one course
    learner: Learner
    course: Course
    request_id: Optional[str] = None


    def __post_init__(self):
        if self.request_id is None:
            learner_ref = getattr(self.learner, "learner_id", "unknown")
            course_ref = getattr(self.course, "course_code", "unknown")
            self.request_id = f"{learner_ref}->{course_ref}"


@dataclass
class RegistrationResult:
    request: RegistrationRequest
    status: str
    message: str
    registration: Optional[Registration] = None
    processed_at: datetime = field(default_factory=lambda: datetime.now(south_africa_tz))

    @property
    def is_success(self) -> bool:
        return self.status == "SUCCESS"

    def __repr__(self):
        return f"<RegistrationResult {self.request.request_id}: {self.status}>"



class RegistrationProcessingEngine:
    # Process a batch of reg requests, enforcing:
    # basic request validation
    # duplicate regestration prevention
    # couse capacity limits

    def __init__(self, simulated_io_delay: float = 0.0):
        self._results: List[RegistrationResult] = []
        """
        Deliverable 3.3
        simulated_io_delay: Seconds to wait before each registration is processed.
        Use this to simulate a slow external API call (like checking a student information system for duplicates). 
        It defaults to 0.0, so it doesn't affect anything unless you explicitly set it. 
        This is mainly here for the Deliverable 3.3 benchmark to show how concurrent processing handles I/O bottlenecks.
        """
        self._simulated_io_delay = simulated_io_delay

    def _simulate_io_wait(self):
        if self._simulated_io_delay > 0:
            time.sleep(self._simulated_io_delay)

    def _validate_request(self, request: RegistrationRequest) -> Optional[str]:
        if request.learner is None:
            return "Request has no learner"
        if not isinstance(request.learner, Learner):
            return f"Expected a Learner instance, got {type(request.learner).__name__}."
        if request.course is None:
            return "Request has no course."
        if not isinstance(request.course, Course):
            return f"Expected a Course instance, got {type(request.course).__name__}."
        return None

    def process_single(self, request: RegistrationRequest) -> RegistrationResult:
        # processes one reg request making sure of business rules
        validation_error = self._validate_request(request)
        if validation_error:
            result = RegistrationResult(request=request, status="FAILED", message=validation_error)
            self._results.append(result)
            return result

        self._simulate_io_wait()  # simulated external eligibility/verification call

        learner = request.learner
        course = request.course
        try:
            registration = learner.register_for_course(course)
        except ValueError as e:
            result = RegistrationResult(request=request, status="FAILED", message=str(e))
        else:
            result = RegistrationResult(
                request=request,
                status="SUCCESS",
                message=f"{learner.name} registered successfully for {course.course_code}.",
                registration=registration,
            )
        self._results.append(result)
        return result

    def process_batch(self, requests: List[RegistrationRequest]) -> "ProcessingSummary":
        for request in requests:
            self.process_single(request)
        return self.get_summary()
    
    def get_summary(self) -> "ProcessingSummary":
        return ProcessingSummary(list(self._results))

    def clear(self):
        self._results = []

    @property
    def results(self) -> List[RegistrationResult]:
        return list(self._results)


class ProcessingSummary:
    # Aggs a set of Registration resul objects into totals and makes a table

    def __init__(self, results: List[RegistrationResult]):
        self.results = results

    @property
    def total(self) -> int:
        return len(self.results)

    @property
    def successful(self) -> List[RegistrationResult]:
        return [r for r in self.results if r.is_success]

    @property
    def failed(self) -> List[RegistrationResult]:
        return [r for r in self.results if not r.is_success]

    @property
    def success_count(self) -> int:
        return len(self.successful)

    @property
    def failure_count(self) -> int:
        return len(self.failed)

    @property
    def success_rate(self) -> float:
        if self.total == 0:
            return 0.0
        return round((self.success_count / self.total) * 100, 2)

    def print_report(self):
        print("=" * 70)
        print("REGISTRATION PROCESSING ENGINE")
        print("=" * 70)
        for result in self.results:
            tag = "[SUCCESS]" if result.is_success else "[FAILED] "
            print(f"{tag} {result.message}")

        print()
        print("REGISTRATION SUMMARY")
        print("-" * 70)
        print(f"Total Registrations Processed: {self.total}")
        print(f"Successful: {self.success_count}")
        print(f"Failed: {self.failure_count}")
        print(f"Success Rate: {self.success_rate}%")

    def __repr__(self):
        return (
            f"<ProcessingSummary total={self.total} "
            f"success={self.success_count} failed={self.failure_count}>"
        )


# Deliverable 2.3 Concurrent processsing
import threading
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

class ConcurrentRegistrationProcessingEngine(RegistrationProcessingEngine):
    """
    I've extended the RegistrationProcessingEngine to handle requests concurrently using a thread pool. 
    To prevent race conditions and over-capacity enrollments, 
    I implemented course-specific locks that ensure "check-then-act" operations are atomic. 
    I also added a separate lock for the internal results list to safely handle concurrent updates from worker threads.
    """

    def __init__(self, max_workers: int = 10, simulated_io_delay: float = 0.0):
        super().__init__(simulated_io_delay=simulated_io_delay)
        self.max_workers = max_workers
        self._course_locks = defaultdict(threading.Lock)
        self._course_locks_guard = threading.Lock()
        self._results_lock = threading.Lock()

    def _get_course_lock(self, course_code: str) -> threading.Lock:
        with self._course_locks_guard:
            return self._course_locks[course_code]

    def process_single(self, request: RegistrationRequest) -> RegistrationResult:
        validation_error = self._validate_request(request)
        if validation_error:
            result = RegistrationResult(request=request, status="FAILED", message=validation_error)
            with self._results_lock:
                self._results.append(result)
            return result

        course_lock = self._get_course_lock(request.course.course_code)

        with course_lock:
            try:
                registration = request.learner.register_for_course(request.course)
            except ValueError as e:
                result = RegistrationResult(request=request, status="FAILED", message=str(e))
            else:
                result = RegistrationResult(
                    request=request,
                    status="SUCCESS",
                    message=(
                        f"{request.learner.name} registered successfully for "
                        f"{request.course.course_code} (concurrent)."
                    ),
                    registration=registration,
                )

        with self._results_lock:
            self._results.append(result)
        return result

    def process_batch_concurrent(self, requests: List[RegistrationRequest]) -> "ProcessingSummary":
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            futures = [executor.submit(self.process_single, request) for request in requests]
            for future in as_completed(futures):
                future.result()

        return self.get_summary()

# Deliverable 2 Output

In [26]:
def deliverable2_1():
    print(f'\n\n{"=" * 70}')
    print("REGISTRATION PROCESSING ENGINE DEMONSTRATION")
    print(f'{"=" * 70}\n')

    course = Course("PY701", "Enterprise Python Development", capacity=10)
    engine = RegistrationProcessingEngine()

    requests = []
    for i in range(17):  # 17 requests against a 10-capacity course
        learner = Learner(f"Learner {i + 1}", f"learner{i + 1}@example.com")
        requests.append(RegistrationRequest(learner=learner, course=course))

    summary = engine.process_batch(requests)
    summary.print_report()


def deliverable2_3():
    print(f'\n\n{"=" * 70}')
    print("CONCURRENT REGISTRATION PROCESSING ENGINE DEMONSTRATION")
    print(f'{"=" * 70}\n')

    course = Course("PY702", "Enterprise Python Development (Concurrent)", capacity=10)
    engine = ConcurrentRegistrationProcessingEngine(max_workers=20)

    requests = [
        RegistrationRequest(
            Learner(f"Concurrent Learner {i + 1}", f"concurrent.learner{i + 1}@example.com"),
            course,
        )
        for i in range(17)  # 17 simultaneous requests against a 10-seat course
    ]

    summary = engine.process_batch_concurrent(requests)
    summary.print_report()

    print(f"\nFinal enrolled_count on Course: {course.enrolled_count} "
          f"(capacity: {course.capacity}) - invariant held under concurrent load.")


deliverable2_1()
deliverable2_3()



REGISTRATION PROCESSING ENGINE DEMONSTRATION

REGISTRATION PROCESSING ENGINE
[SUCCESS] Learner 1 registered successfully for PY701.
[SUCCESS] Learner 2 registered successfully for PY701.
[SUCCESS] Learner 3 registered successfully for PY701.
[SUCCESS] Learner 4 registered successfully for PY701.
[SUCCESS] Learner 5 registered successfully for PY701.
[SUCCESS] Learner 6 registered successfully for PY701.
[SUCCESS] Learner 7 registered successfully for PY701.
[SUCCESS] Learner 8 registered successfully for PY701.
[SUCCESS] Learner 9 registered successfully for PY701.
[SUCCESS] Learner 10 registered successfully for PY701.
[FAILED]  Course PY701 is full. annot register Learner 11
[FAILED]  Course PY701 is full. annot register Learner 12
[FAILED]  Course PY701 is full. annot register Learner 13
[FAILED]  Course PY701 is full. annot register Learner 14
[FAILED]  Course PY701 is full. annot register Learner 15
[FAILED]  Course PY701 is full. annot register Learner 16
[FAILED]  Course PY701

# Deliverable 3

In [27]:
"""
Bugzot monitoring subsystem 9event logger)

Records events 9sucess, validation, failures etc) with enough comtext

all timestamps are in SAST = UCT + 2
"""

from dataclasses import dataclass, field
from datetime import datetime, timezone, timedelta
from enum import Enum
from typing import List, Optional, Dict, Any

south_africa_tz = timezone(timedelta(hours=2))

class EventLevel(Enum):
    INFO = "INFO"
    WARNING = "WARNING"
    ERROR = "ERROR"

class EventCategory(Enum):
    REGISTRATION_SUCCESS = "Registration Success"
    VALIDATION_FAILURE = "Validation Failure"
    DUPLICATE_REGISTRATION = "Duplicate Registration"
    CAPACITY_VIOLATION = "Capacity Violation"
    APPLICATION_ERROR = "Application Error"


@dataclass
class BugzotEvent:
    # a single recorded event with enough context
    level: EventLevel
    category: EventCategory
    message: str
    source: str = "unknown"
    context: Dict[str, Any] = field(default_factory=dict)
    timestamp: datetime = field(default_factory=lambda: datetime.now(south_africa_tz))

    def format_line(self) -> str:
        ts = self.timestamp.strftime("%Y-%m-%d %H:%M:%S") # year-month-day Hour:Minute:Second
        return f"{ts} | {self.level.value} | {self.category.value} | {self.message}"

    def __repr__(self):
        return f"<BugzotEvent {self.category.value}: {self.message}>"

class BugzotLogger:
    def __init__(self):
        self._events: List[BugzotEvent] = []

    # Recording
    def log(self, level: EventLevel, category: EventCategory, message: str, source: str = "unknown", 
            context: Optional[Dict[str, Any]] = None) -> BugzotEvent:

        event = BugzotEvent(level=level, category=category, message=message, source=source, context=context or {})
        self._events.append(event)
        return event

    def log_success(self, message: str, source: str = "unknown", **context) -> BugzotEvent:
        return self.log(EventLevel.INFO, EventCategory.REGISTRATION_SUCCESS, message, source, context)

    def log_validation_failure(self, message: str, source: str = "unknown", **context) -> BugzotEvent:
        return self.log(EventLevel.ERROR, EventCategory.VALIDATION_FAILURE, message, source, context)

    def log_duplicate_registration(self, message: str, source: str = "unknown", **context) -> BugzotEvent:
        return self.log(EventLevel.WARNING, EventCategory.DUPLICATE_REGISTRATION, message, source, context)

    def log_capacity_violation(self, message: str, source: str = "unknown", **context) -> BugzotEvent:
        return self.log(EventLevel.WARNING, EventCategory.CAPACITY_VIOLATION, message, source, context)

    def log_application_error(self, message: str, source: str = "unknown", **context) -> BugzotEvent:
        return self.log(EventLevel.ERROR, EventCategory.APPLICATION_ERROR, message, source, context)


    def record_from_result(self, result, source: str = "RegistrationProcessingEngine") -> BugzotEvent:
        context = {"request_id": getattr(result.request, "request_id", None)}

        if result.registration is not None:
            context["registration_id"] = result.registration.registration_id
        if result.request.learner is not None:
            context["learner_id"] = getattr(result.request.learner, "learner_id", None)
        if result.request.course is not None:
            context["course_code"] = getattr(result.request.course, "course_code", None)

        if result.is_success:
            return self.log_success(result.message, source=source, **context)

        message_lower = result.message.lower()

        if "already" in message_lower and "registered" in message_lower:
            return self.log_duplicate_registration(result.message, source=source, **context)

        if "full" in message_lower:
            return self.log_capacity_violation(result.message, source=source, **context)

        if any(
            keyword in message_lower
            for keyword in ("has no learner", "has no course", "instance, got")
        ):
            return self.log_validation_failure(result.message, source=source, **context)

        return self.log_application_error(result.message, source=source, **context)



    # Querying
    @property
    def events(self) -> List[BugzotEvent]:
        return list(self._events)

    def events_by_category(self, category: EventCategory) -> List[BugzotEvent]:
        return [e for e in self._events if e.category == category]

    def events_by_level(self, level: EventLevel) -> List[BugzotEvent]:
        return [e for e in self._events if e.level == level]

    def count_by_category(self) -> Dict[str, int]:
        counts: Dict[str, int] = {}
        for event in self._events:
            counts[event.category.value] = counts.get(event.category.value, 0) + 1
        return counts

    def clear(self):
        self._events = []


    # Reporting
    def print_log(self, title: str = "BUGZOT EVENT MONITORING LOG"):
        print("=" * 70)
        print(title)
        print("=" * 70)
        for event in self._events:
            print(event.format_line())
        print()

    def __repr__(self):
        return f"<BugzotLogger events={len(self._events)}>"


import time
import statistics
from contextlib import contextmanager
from dataclasses import dataclass, field
from datetime import datetime, timezone, timedelta
from typing import List, Dict, Optional

south_africa_tz = timezone(timedelta(hours=2))

@dataclass
class TransactionMetric:
    component: str
    operation: str
    duration_ms: float
    success: bool
    timestamp: datetime = field(default_factory=lambda: datetime.now(south_africa_tz))
    detail: Optional[str] = None

    def __repr__(self):
        outcome= "OK" if self.success else "FAILED"
        return f"<TransactionMetric {self.component} {self.operation} {self.duration_ms}ms [{outcome}]"

class PerformanceMonitor:
    def __init__(self):
        self._metrics: List[TransactionMetric] = []

    # recording
    @contextmanager
    def measure(self, component: str, operation: str, detail: Optional[str] = None):
        start = time.perf_counter()
        success = True
        try:
            yield
        except Exception:
            success = False
            raise
        finally:
            duration_ms = (time.perf_counter() - start) * 1000
            self._metrics.append(
                TransactionMetric(
                    component=component,
                    operation=operation,
                    duration_ms=duration_ms,
                    success=success,
                    detail=detail,
                )
            )

    def record_manual(self, component:str, operation:str, duration_ms:float,
                      success: bool = True, detail: Optional[str] = None) -> TransactionMetric:

        metric = TransactionMetric(
            component=component,
            operation=operation,
            duration_ms=duration_ms,
            success=success,
            detail=detail,
        )
        self._metrics.append(metric)
        return metric



    # querying/stats
    @property
    def metrics(self) -> List[TransactionMetric]:
        return list(self._metrics)

    @property
    def transaction_count(self) -> int:
        return len(self._metrics)

    def metrics_for(self, component: Optional[str] = None, operation: Optional[str] = None) -> List[TransactionMetric]:
        result = self._metrics
        if component is not None:
            result = [m for m in result if m.component == component]
        if operation is not None:
            result = [m for m in result if m.operation == operation]
        return result

    def _durations(self, metrics: List[TransactionMetric]) -> List[float]:
        return [m.duration_ms for m in metrics]

    def average_duration_ms(self, component: Optional[str] = None, operation: Optional[str] = None) -> float:
        durations = self._durations(self.metrics_for(component, operation))
        if not durations:
            return 0.0
        return round(statistics.mean(durations), 3)

    def min_duration_ms(self, component: Optional[str] = None, operation: Optional[str] = None) -> float:
        durations = self._durations(self.metrics_for(component, operation))
        return round(min(durations), 3) if durations else 0.0

    def max_duration_ms(self, component: Optional[str] = None, operation: Optional[str] = None) -> float:
        durations = self._durations(self.metrics_for(component, operation))
        return round(max(durations), 3) if durations else 0.0

    def success_rate(self, component: Optional[str] = None, operation: Optional[str] = None) -> float:
        subset = self.metrics_for(component, operation)
        if not subset:
            return 0.0
        successes = sum(1 for m in subset if m.success)
        return round((successes / len(subset)) * 100, 2)



    def throughput_per_second(self, component: Optional[str] = None, 
                              operation: Optional[str] = None) -> float:


        subset = sorted(self.metrics_for(component, operation), key=lambda m: m.timestamp)
        if len(subset) < 2:
            return 0.0

        elapsed_seconds = (subset[-1].timestamp - subset[0].timestamp).total_seconds()
        if elapsed_seconds <= 0:
            elapsed_seconds = sum(self._durations(subset)) / 1000
            if elapsed_seconds <= 0:
                return 0.0

        return round(len(subset) / elapsed_seconds, 2)

    def components(self) -> List[str]:
        return sorted({m.component for m in self._metrics})

    def clear(self):
        self._metrics = []

        
    # reporting
    def generate_report(self) -> Dict:
        report = {
            "total_transactions": self.transaction_count,
            "overall_success_rate": self.success_rate(),
            "overall_average_duration_ms": self.average_duration_ms(),
            "components": {},
        }

        for component in self.components():
            report["components"][component] = {
                "transaction_count": len(self.metrics_for(component=component)),
                "average_duration_ms": self.average_duration_ms(component=component),
                "min_duration_ms": self.min_duration_ms(component=component),
                "max_duration_ms": self.max_duration_ms(component=component),
                "success_rate": self.success_rate(component=component),
                "throughput_per_second": self.throughput_per_second(component=component),
            }

        return report


    def print_report(self, title: str = "APPLICATION PERFORMANCE REPORT"):
        report = self.generate_report()

        print("=" * 70)
        print(title)
        print("=" * 70)
        print(f"Total Transactions Recorded: {report['total_transactions']}")
        print(f"Overall Success Rate: {report['overall_success_rate']}%")
        print(f"Overall Average Duration: {report['overall_average_duration_ms']}ms")
        print()

        for component, stats in report["components"].items():
            print(f"Component: {component}")
            print("-" * 70)
            print(f"  Transactions: {stats['transaction_count']}")
            print(f"  Average Duration: {stats['average_duration_ms']}ms")
            print(f"  Min Duration: {stats['min_duration_ms']}ms")
            print(f"  Max Duration: {stats['max_duration_ms']}ms")
            print(f"  Success Rate: {stats['success_rate']}%")
            print(f"  Throughput: {stats['throughput_per_second']} transactions/sec")
            print()

    def __repr__(self):
        return f"<PerformanceMonitor transactions={self.transaction_count}>"

# Deliverable 3 Output

In [28]:
def deliverable3_1():
    print(f'\n\n{"=" * 70}')
    print("BUGZOT MONITORING SUBSYSTEM DEMONSTRATION")
    print(f'{"=" * 70}\n')

    bugzot = BugzotLogger()
    engine = RegistrationProcessingEngine()

    course = Course("PY703", "Enterprise Python Development (Bugzot)", capacity=5)

    valid_learner = Learner("Anele Dlamini", "anele@example.com")
    duplicate_learner = Learner("Sipho Mokoena", "sipho@example.com")

    scenarios = [
        RegistrationRequest(valid_learner, course),
        RegistrationRequest(duplicate_learner, course),
        RegistrationRequest(duplicate_learner, course),        # duplicate
        RegistrationRequest(None, course),                     # validation failure
        RegistrationRequest(Learner("Lerato Nkosi", "lerato@example.com"), course),
        RegistrationRequest(Learner("Thando Maseko", "thando@example.com"), course),
        RegistrationRequest(Learner("Naledi Khumalo", "naledi.b@example.com"), course),
        RegistrationRequest(Learner("Over Capacity Learner", "overcap@example.com"), course),  # capacity violation
    ]

    for request in scenarios:
        result = engine.process_single(request)
        bugzot.record_from_result(result)

    bugzot.print_log()

    print("EVENT COUNTS BY CATEGORY")
    print("-" * 60)
    for category, count in bugzot.count_by_category().items():
        print(f"{category}: {count}")


def deliverable3_2():
    print(f'\n\n{"=" * 70}')
    print("BUGZOT APPLICATION PERFORMANCE MONITORING DEMONSTRATION")
    print(f'{"=" * 70}\n')

    monitor = PerformanceMonitor()
    engine = RegistrationProcessingEngine()
    course = Course("PY704", "Enterprise Python Development (Performance)", capacity=12)

    for i in range(15):
        learner = Learner(f"Perf Learner {i + 1}", f"perf.learner{i + 1}@example.com")
        request = RegistrationRequest(learner, course)

        with monitor.measure("RegistrationProcessingEngine", "process_single", detail=request.request_id):
            engine.process_single(request)

    monitor.print_report()


def deliverable3_3():
    print(f'\n\n{"=" * 70}')
    print("PERFORMANCE IMPROVEMENT: SEQUENTIAL vs CONCURRENT PROCESSING")
    print(f'{"=" * 70}\n')

    SIMULATED_DELAY = 0.05   # 50ms - representative of a fast external verification call
    NUM_LEARNERS = 30
    NUM_COURSES = 6           # spread across several courses, as would be typical at peak enrolment

    def build_requests(code_prefix):
        courses = [Course(f"{code_prefix}{i}", f"Course {i}", capacity=20) for i in range(NUM_COURSES)]
        requests = []
        for i in range(NUM_LEARNERS):
            learner = Learner(f"{code_prefix} Learner {i + 1}", f"{code_prefix.lower()}.learner{i + 1}@example.com")
            course = courses[i % NUM_COURSES]
            requests.append(RegistrationRequest(learner, course))
        return requests

    monitor = PerformanceMonitor()

    # --- BEFORE: sequential processing ---
    sequential_engine = RegistrationProcessingEngine(simulated_io_delay=SIMULATED_DELAY)
    sequential_requests = build_requests("SEQ")
    with monitor.measure("RegistrationProcessingEngine", "process_batch (sequential)"):
        sequential_summary = sequential_engine.process_batch(sequential_requests)

    # --- AFTER: concurrent processing ---
    concurrent_engine = ConcurrentRegistrationProcessingEngine(max_workers=10, simulated_io_delay=SIMULATED_DELAY)
    concurrent_requests = build_requests("CON")
    with monitor.measure("ConcurrentRegistrationProcessingEngine", "process_batch_concurrent (after fix)"):
        concurrent_summary = concurrent_engine.process_batch_concurrent(concurrent_requests)

    monitor.print_report()

    sequential_ms = monitor.metrics_for(operation="process_batch (sequential)")[0].duration_ms
    concurrent_ms = monitor.metrics_for(operation="process_batch_concurrent (after fix)")[0].duration_ms
    speedup = round(sequential_ms / concurrent_ms, 2) if concurrent_ms > 0 else float("inf")

    print("SUMMARY")
    print("-" * 70)
    print(f"Requests processed: {NUM_LEARNERS} (across {NUM_COURSES} courses)")
    print(f"Simulated I/O delay per request: {SIMULATED_DELAY * 1000:.0f}ms")
    print(f"BEFORE (sequential): {sequential_ms:.2f}ms total, {sequential_summary.success_count} succeeded")
    print(f"AFTER  (concurrent): {concurrent_ms:.2f}ms total, {concurrent_summary.success_count} succeeded")
    print(f"Speedup: {speedup}x")

deliverable3_1()
deliverable3_2()
deliverable3_3()



BUGZOT MONITORING SUBSYSTEM DEMONSTRATION

BUGZOT EVENT MONITORING LOG
2026-08-26 12:31:43 | INFO | Registration Success | Anele Dlamini registered successfully for PY703.
2026-08-26 12:31:43 | INFO | Registration Success | Sipho Mokoena registered successfully for PY703.
2026-08-26 12:31:43 | WARNING | Duplicate Registration | Learner L038 is already registered for PY703
2026-08-26 12:31:43 | ERROR | Validation Failure | Request has no learner
2026-08-26 12:31:43 | INFO | Registration Success | Lerato Nkosi registered successfully for PY703.
2026-08-26 12:31:43 | INFO | Registration Success | Thando Maseko registered successfully for PY703.
2026-08-26 12:31:43 | INFO | Registration Success | Naledi Khumalo registered successfully for PY703.
2026-08-26 12:31:43 | WARNING | Capacity Violation | Course PY703 is full. annot register Over Capacity Learner

EVENT COUNTS BY CATEGORY
------------------------------------------------------------
Registration Success: 5
Duplicate Registration: